In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is inquiring about various fictional aspects of Lunapolis, the capital of the moon, including its weather and the cheese miners' community and their potential for a union strike. \n\n## SUMMARY\n- The capital of the moon is identified as Lunapolis.\n- Current weather in Lunapolis is reported to be clear skies, with temperatures ranging from a high of 120C to a low of -100C.\n- There are approximately 100,000 cheese miners residing in Lunapolis.\n- A potential strike by the cheese miners' union is anticipated due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='08a5930f-06b6-4641-91ff-557ec486403b'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metad

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is inquiring about various fictional aspects of Lunapolis, the capital of the moon, including its weather and the cheese miners' community and their potential for a union strike. 

## SUMMARY
- The capital of the moon is identified as Lunapolis.
- Current weather in Lunapolis is reported to be clear skies, with temperatures ranging from a high of 120C to a low of -100C.
- There are approximately 100,000 cheese miners residing in Lunapolis.
- A potential strike by the cheese miners' union is anticipated due to dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
None


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='a3dba058-8be4-4458-a894-4a87a1968d60'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='422a3ec7-38f7-45f6-bc16-44e582faf427', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='80d001b5-ea42-43eb-823b-62b5694bc405'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='876a0371-e808-44a3-ac6a-faed67094e98', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='fbf08f31-d053-44df-b68a-02658911b55d'),
              AIMessage(content='I can’t read the device’s temperature from here. If you want t

In [8]:
print(response["messages"][-1].content)

I can’t read the device’s temperature from here. If you want to check it yourself, tell me what kind of device you’re using (laptop, desktop PC, phone, tablet, router, etc.) and the model. In the meantime, try these general steps:

- If the device is hot to the touch: power it off, unplug it, and let it cool on a non-flammable surface for 15–30 minutes. Make sure vents are clear and nothing is blocking fans.
- Check for indicators: are any lights on? do you hear fans spinning or any beeps? This can help diagnose power issues.
- Power/reset steps:
  - Do a simple reset: unplug (and remove battery if possible), hold the power button for 15–20 seconds, reconnect, and try powering on again.
  - Try a different power outlet or a different charger/adapter if you have one compatible.
- If you can boot enough to see something:
  - On Windows: Task Manager > Performance > CPU to view temps (you might need a third-party app like HWInfo or Core Temp for full readings).
  - On macOS: Activity Moni